# Learn Early, Exploit Late — reproducibility notebook

This notebook verifies frozen inputs and hashes, loads real summaries and traces, selects the comparator from development results, reconstructs secondary metrics and claim gates, and reproduces the staged counterexample. It prints `NOT EVALUABLE` when required evidence is missing; it never substitutes placeholder scores. Public ARC-AGI-3 games are controlled engineering evidence, not an unseen-generalization set.

In [ ]:
from collections import defaultdict
from dataclasses import asdict, replace
from pathlib import Path
from statistics import mean
import hashlib
import json
import os

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'run from the repository or notebooks directory'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('repository:', ROOT)
print('uv.lock sha256:', sha256(ROOT / 'uv.lock'))

In [ ]:
from arc3_voi.config import load_config
from arc3_voi.experiment import stable_config_hash
from arc3_voi.splitting import load_metadata, metadata_hash, stratified_split

selection_path = ROOT / 'artifacts/selected_config.json'
if selection_path.exists():
    selection = read_json(selection_path)
    config_path = ROOT / selection['development_config']
else:
    config_path = Path(os.environ.get('ARC3_REPRO_CONFIG', ROOT / 'configs/local_4b.yaml'))
    print('selected config is provisional; freeze artifacts/selected_config.json before runs')
config = load_config(config_path)
snapshot_path = ROOT / 'artifacts/public_games.snapshot.json'
split_path = ROOT / 'artifacts/public_split.json'
matrix_path = ROOT / 'artifacts/development_matrix.json'
for required in (snapshot_path, split_path, matrix_path):
    assert required.exists(), f'missing frozen artifact: {required}'

games = load_metadata(snapshot_path)
snapshot = read_json(snapshot_path)
split = read_json(split_path)
recomputed = stratified_split(games, development_size=15, seed=20260712)
assert metadata_hash(games) == snapshot['metadata_hash'] == split['metadata_hash']
assert tuple(split['development']) == recomputed.development
assert tuple(split['confirmation']) == recomputed.confirmation
matrix = read_json(matrix_path)
assert len(matrix) == 180
assert {row['variant'] for row in matrix} == {'D', 'S', 'M', 'X'}
assert {row['seed'] for row in matrix} == {11, 23, 47}
assert {row['game_id'] for row in matrix} == set(split['development'])
expected_config_hashes = {variant: stable_config_hash(
    replace(config, experiment=replace(config.experiment, variant=variant)))
    for variant in ('D', 'S', 'M', 'X')}
matrix_hash_ok = all(row['config_hash'] == expected_config_hashes[row['variant']] for row in matrix)
print({'config_sha256': stable_config_hash(config), 'games': len(games),
       'development': len(split['development']), 'confirmation': len(split['confirmation']),
       'development_runs': len(matrix), 'matrix_config_hashes_verified': matrix_hash_ok})
if not matrix_hash_ok:
    print('development matrix: STALE - regenerate before running or reporting experiments')

In [ ]:
# Optional offline-bundle verification uses exactly the release manifest.
from scripts.build_offline_bundle import verify_manifest

bundle = ROOT / 'build/kaggle-bundle'
if bundle.exists():
    manifest = verify_manifest(bundle)
    print('bundle verified:', len(manifest['files']), 'payload files')
else:
    print('bundle: NOT EVALUABLE (build/kaggle-bundle is absent)')

preflight_path = ROOT / 'artifacts/local_4b_preflight.json'
if preflight_path.exists():
    preflight = read_json(preflight_path)
    print('checked preflight artifact:', preflight)
    print('local model gate:', bool(preflight.get('fits_vram_gate')) and
          bool(preflight.get('passes_throughput_gate')))
else:
    print('local model gate: NOT EVALUABLE (preflight artifact absent)')
kaggle_preflight_path = ROOT / 'artifacts/kaggle_model_preflight.json'
if kaggle_preflight_path.exists():
    kaggle_preflight = read_json(kaggle_preflight_path)
    print('Kaggle model/runtime gate:', bool(kaggle_preflight.get('fits_vram_gate')) and
          bool(kaggle_preflight.get('passes_runtime_gate')))
else:
    print('Kaggle model/runtime gate: NOT EVALUABLE')
submission_path = ROOT / 'artifacts/kaggle_submission.json'
print('linked private submission:', read_json(submission_path) if submission_path.exists() else
      'NOT EVALUABLE (no submission artifact)')

In [ ]:
# Load only actual per-run summary JSON files and their matching JSONL traces.
from arc3_voi.metrics import load_run

run_dir = ROOT / 'artifacts/runs'
runs = []
run_objects = []
for path in sorted(run_dir.glob('*.json')) if run_dir.exists() else []:
    row = read_json(path)
    required = {'run_id', 'game_id', 'seed', 'variant'}
    if not required.issubset(row):
        continue
    trace_path = path.with_suffix('.jsonl')
    trace = []
    if trace_path.exists():
        trace = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    losses = [float(step['weighted_transition_loss']) for step in trace if step.get('weighted_transition_loss') is not None]
    decisions = len(trace)
    row['_mean_prequential_loss'] = mean(losses) if losses else None
    row['_valid_pool_rate'] = (sum(step.get('valid_hypotheses', 0) >= 2 for step in trace) / decisions) if decisions else None
    row['_fallback_decision_rate'] = (sum(bool(step.get('fallback')) for step in trace) / decisions) if decisions else None
    row['_best_hypothesis_loss'] = mean(
        [float(step['best_hypothesis_transition_loss']) for step in trace
         if step.get('best_hypothesis_transition_loss') is not None]) if any(
        step.get('best_hypothesis_transition_loss') is not None for step in trace) else None
    row['_timeout_rate'] = (float(row['program_timeouts']) / float(row['program_prediction_calls'])
                            if row.get('timeout_instrumentation_complete') and
                            row.get('program_prediction_calls') else None)
    runs.append(row)
    run_objects.append(load_run(path, trace_path))
print('real run summaries:', len(runs))
if not runs:
    print('results: NOT EVALUABLE - this notebook does not fabricate placeholder scores')

In [ ]:
# Secondary-metric table. Missing fields remain None rather than becoming zero.
secondary = ('rhae', 'levels_completed', 'total_actions', 'generated_tokens',
             'wall_seconds', 'peak_vram_gb', '_mean_prequential_loss',
             '_best_hypothesis_loss', '_valid_pool_rate', '_timeout_rate',
             '_fallback_decision_rate')
table = []
for variant in ('D', 'S', 'M', 'X'):
    selected = [row for row in runs if row['variant'] == variant]
    if not selected:
        continue
    game_rows = defaultdict(list)
    for row in selected:
        game_rows[row['game_id']].append(row)
    summary = {'variant': variant, 'runs': len(selected), 'games': len(game_rows)}
    for field in secondary:
        per_game = []
        for replicas in game_rows.values():
            values = [float(row[field]) for row in replicas if row.get(field) is not None]
            if values:
                per_game.append(mean(values))
        summary[field] = mean(per_game) if per_game else None
    table.append(summary)
table

In [ ]:
# Select the confirmation comparator from development evidence; never hard-code M.
from arc3_voi.metrics import evaluate_development_score_gate, evaluate_mechanism_gate
from arc3_voi.statistics import (ScoreObservation, confirmation_claim_passes,
    paired_game_deltas, strongest_comparator, summarize_paired_observations)

observations = [ScoreObservation(row['game_id'], int(row['seed']), row['variant'], float(row['rhae']))
                for row in runs if row.get('rhae') is not None]
development = [row for row in runs if row['run_id'].startswith('development-') and row.get('rhae') is not None]
development_obs = [ScoreObservation(row['game_id'], int(row['seed']), row['variant'], float(row['rhae']))
                   for row in development]
replicate_counts = defaultdict(int)
for row in development:
    replicate_counts[(row['game_id'], row['variant'])] += 1
development_complete = (matrix_hash_ok and len(development) == 180 and len(replicate_counts) == 60 and
                        set(replicate_counts.values()) == {3} and
                        {row['variant'] for row in development} == {'D', 'S', 'M', 'X'} and
                        all(row.get('config_hash') == expected_config_hashes[row['variant']]
                            for row in development))
comparator = strongest_comparator(development_obs) if development_complete else None
print('development-selected comparator:', comparator or 'NOT EVALUABLE')

score_gate = None
if comparator:
    deltas = paired_game_deltas(development_obs, 'X', 'M')
    by_variant = defaultdict(list)
    for row in development:
        by_variant[row['variant']].append(row)
    if development_complete and len(deltas) == 15 and by_variant['X'] and by_variant['M']:
        development_objects = [run for run in run_objects if run.run_id.startswith('development-')]
        development_gate = evaluate_development_score_gate(development_objects)
        score_gate = development_gate.gate
        print('development score gate:', asdict(development_gate))
    else:
        print('development score gate: NOT EVALUABLE (incomplete 15-game matrix)')
else:
    print('development score gate: NOT EVALUABLE')

confirmation = [row for row in runs if row['run_id'].startswith('confirmation-') and row.get('rhae') is not None]
confirmation_counts = defaultdict(int)
for row in confirmation:
    confirmation_counts[(row['game_id'], row['variant'])] += 1
confirmation_complete = (len(confirmation) == 100 and len(confirmation_counts) == 20 and
                         set(confirmation_counts.values()) == {5} and comparator is not None and
                         {row['variant'] for row in confirmation} == {comparator, 'X'})
if comparator and confirmation_complete:
    confirmation_obs = [ScoreObservation(row['game_id'], int(row['seed']), row['variant'], float(row['rhae'])) for row in confirmation]
    confirmation_deltas = paired_game_deltas(confirmation_obs, 'X', comparator)
    if len(confirmation_deltas) == 10:
        paired = summarize_paired_observations(confirmation_obs, 'X', comparator)
        print(asdict(paired))
        print('confirmation claim gate:', confirmation_claim_passes(paired))
    else:
        print('confirmation claim gate: NOT EVALUABLE (incomplete 10-game pairing)')
else:
    print('confirmation claim gate: NOT EVALUABLE')

committee_objects = [run for run in run_objects
                     if run.run_id.startswith('development-') and run.variant == 'X']
single_objects = [run for run in run_objects
                  if run.run_id.startswith('development-') and run.variant == 'S']
if committee_objects and single_objects:
    print('mechanism gate:', asdict(evaluate_mechanism_gate(committee_objects, single_objects)))
else:
    print('mechanism gate: NOT EVALUABLE (X/S traces absent)')

In [ ]:
# Finite deterministic staged counterexample from docs/theory.md.
myopic_evsi = (1 + 2 + 2) / 3 - 1
no_probe = 1 * ((1 + 2 + 2) / 3) + 2 * ((1/3) * 1 + (2/3) * ((1 + 2) / 2)) + 3 * 1
probe = 1 * 2 + 2 * 1 + 3 * 1
m1 = 1 + (2 + 3) / 1
assert myopic_evsi - 1 < 0
assert m1 * myopic_evsi - 1 > 0
from math import isclose
assert isclose(no_probe, 22 / 3) and probe == 7 and isclose(no_probe - probe, 1 / 3)
{'EVSI': myopic_evsi, 'myopic_utility': myopic_evsi - 1,
 'cross_level_utility': m1 * myopic_evsi - 1,
 'actual_weighted_saving': no_probe - probe}

In [ ]:
# ARC-AGI-2 is conditional and has no result unless the ARC-AGI-3 score gate passed first.
arc2_path = ROOT / 'artifacts/arc2/results.json'
if score_gate is None or not score_gate.passed:
    print('ARC-AGI-2: GATED OFF; no result is reportable')
elif not arc2_path.exists():
    print('ARC-AGI-2: authorized by score gate but NOT EVALUATED')
else:
    arc2 = read_json(arc2_path)
    assert arc2.get('evaluation_tasks') == 120 and arc2.get('evaluation_runs') == 1
    print(arc2)

print('Reporting contract: games, not actions, levels, or episodes, are the paired units.')